# Re-annoter le 800k (externe) et le 110k avec le prof bidoua-aware -- via rclone (sans drive.mount)

**Pourquoi rclone et pas `drive.mount()`** : le flux d'authentification
`dfs_ephemeral` de Colab echoue (400 `credentials-propagation`) sur ce
compte. On contourne en accedant a Drive via **rclone** avec le meme
token OAuth deja configure en local pour ce projet -- independant du flux
casse de Colab.

Principe (different de la version drive.mount) :
1. rclone est configure sur Colab a partir de TON token (colle via un
   champ masque, jamais ecrit en clair dans le notebook).
2. Les datasets d'entree sont **telecharges** depuis Drive vers le disque
   local Colab (rapide).
3. L'annotation tourne sur le disque local Colab. Le cache SQLite + les
   sorties sont **resynchronises periodiquement vers Drive** en tache de
   fond (pour survivre aux deconnexions, comme le faisait le mount).
4. En cas de deconnexion : re-executer les cellules dans l'ordre --
   l'etape 5 re-tire le cache depuis Drive, et l'annotation reprend
   exactement ou elle s'etait arretee (le cache deduplique le deja-fait).

**ATTENTION CACHE (piege reel)** : le cache d'annotations est indexe par
position + config du professeur, PAS par version du code d'evaluation.
Tous les dossiers de cache ci-dessous portent un suffixe `_bidoua_v1` :
ne jamais les faire pointer vers un ancien cache (pre-correctif), sinon
on recupere silencieusement les anciennes annotations sans bidoua.

**Debit a mesurer sur CE Colab avant le volume complet** (ne pas
supposer) : ~15-16s/position mesure autrefois sur l'externe, ~61s/position
mesure en local sur le 110k (budget de 60s presque toujours epuise). Deux
chiffres non interchangeables -> toujours piloter d'abord.


## 1 -- Configurer rclone sur Colab (upload de ton rclone.conf)

**Aucun secret a taper dans une cellule** (une erreur de collage
exposerait le token). On uploade directement le fichier de config rclone.

Sur ton Mac, ce fichier est a `~/.config/rclone/rclone.conf`. Comme
`~/.config` est un dossier cache, dans le selecteur de fichier qui va
s'ouvrir : appuie sur **Cmd+Shift+.** pour afficher les fichiers caches,
puis navigue jusqu'a `~/.config/rclone/rclone.conf`. (Si tu ne le trouves
pas, copie-le d'abord sur ton Bureau : `cp ~/.config/rclone/rclone.conf
~/Desktop/` -- uploade-le, puis supprime la copie du Bureau.)

**Si tu viens de regenerer ton token** (revocation + `rclone config
reconnect songo_drive:` apres la fuite) : c'est bien le NOUVEAU
`rclone.conf` qu'il faut uploader.

In [ ]:
import os, subprocess
from google.colab import files

# Installer le binaire rclone sur Colab
subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True, check=False)

# Uploader ton rclone.conf local (selecteur de fichier -- aucun secret tape ici)
print("Selectionne ton fichier local ~/.config/rclone/rclone.conf ...")
uploaded = files.upload()
assert uploaded, "aucun fichier selectionne"
name = list(uploaded)[0]
content = uploaded[name]
assert b"[songo_drive]" in content, "ce fichier ne contient pas le remote [songo_drive] : mauvais fichier ?"

os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "wb") as f:
    f.write(content)
os.chmod("/root/.config/rclone/rclone.conf", 0o600)
del content, uploaded  # ne pas garder en memoire

# Test : doit lister le contenu de SongoFish/data sur ton Drive
print(subprocess.run("rclone lsd songo_drive:SongoFish/data", shell=True,
                     capture_output=True, text=True).stdout)


AssertionError: Token invalide : doit commencer par { et contenir access_token

## 2 -- Cloner le depot et installer

Branche `dev` (le correctif bidoua n'est pas encore sur `main`).

In [ ]:
!git clone -b dev https://github.com/GlennEriss/songo-fish.git /content/songo-fish
%cd /content/songo-fish
!pip install -q -e ".[train]"


In [ ]:
import multiprocessing
print('coeurs disponibles sur cette session Colab:', multiprocessing.cpu_count())


## 3 -- Chemins + telechargement des entrees + fonctions de synchro

Definit les chemins (Drive distant + miroir local Colab), telecharge les
datasets d'entree, re-tire les caches deja constitues (reprise), et
definit `start_bg_sync()` / `stop_bg_sync()` pour la resync automatique
vers Drive pendant les longues annotations.

In [ ]:
import subprocess, os, signal

# --- Distant (Drive, via rclone) ---
REMOTE = "songo_drive:SongoFish/data"
R_EXTERNAL_IN  = f"{REMOTE}/dataset_full_matrix_merged_all_colabs"
R_EXTERNAL_OUT = f"{REMOTE}/dataset_external_reannotated_bidoua_v1"
R_OWN_IN       = f"{REMOTE}/dataset_v003_110k"
R_OWN_OUT      = f"{REMOTE}/dataset_v003_110k_bidoua_v1"

# --- Local (disque Colab, ephemere) ---
L = "/content/songo_data"
L_EXTERNAL_IN  = f"{L}/dataset_full_matrix_merged_all_colabs"
L_EXTERNAL_OUT = f"{L}/dataset_external_reannotated_bidoua_v1"
L_EXTERNAL_CACHE = f"{L_EXTERNAL_OUT}/annotation_cache"
L_OWN_IN       = f"{L}/dataset_v003_110k"
L_OWN_OUT      = f"{L}/dataset_v003_110k_bidoua_v1"
L_OWN_CACHE    = f"{L_OWN_OUT}/annotation_cache"

for d in (L_EXTERNAL_OUT, L_EXTERNAL_CACHE, L_OWN_OUT, L_OWN_CACHE):
    os.makedirs(d, exist_ok=True)

def rc(cmd):
    print(">", cmd)
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

# Telecharger les entrees Drive -> local (rapide, une fois)
rc(f'rclone copy "{R_EXTERNAL_IN}" "{L_EXTERNAL_IN}" --progress')
rc(f'rclone copy "{R_OWN_IN}" "{L_OWN_IN}" --progress')

# Reprise : re-tirer les caches + sorties deja constitues sur Drive (no-op au 1er lancement)
rc(f'rclone copy "{R_EXTERNAL_OUT}" "{L_EXTERNAL_OUT}"')
rc(f'rclone copy "{R_OWN_OUT}" "{L_OWN_OUT}"')

# --- Synchro periodique local -> Drive, en tache de fond ---
_sync_proc = None
def start_bg_sync(local_dir, remote_dir, every_s=300):
    global _sync_proc
    stop_bg_sync()
    script = (f'while true; do sleep {every_s}; '
              f'rclone copy "{local_dir}" "{remote_dir}" >/dev/null 2>&1; done')
    _sync_proc = subprocess.Popen(["bash", "-c", script], preexec_fn=os.setsid)
    print(f"[sync] tache de fond demarree : {local_dir} -> {remote_dir} toutes les {every_s}s")

def stop_bg_sync():
    global _sync_proc
    if _sync_proc is not None:
        os.killpg(os.getpgid(_sync_proc.pid), signal.SIGTERM)
        _sync_proc = None

def push(local_dir, remote_dir):
    rc(f'rclone copy "{local_dir}" "{remote_dir}"')
    print("[sync] push final termine")


## Section A -- dataset externe (883k, vraies parties)

### A.1 -- pilote (mesurer le debit reel sur CE Colab)

In [ ]:
!.venv/bin/python apps/trainer/scripts/reannotate_external_dataset.py \
    --input-dir "{L_EXTERNAL_IN}" \
    --out-dir /content/pilot_external_bidoua \
    --cache-dir "{L_EXTERNAL_CACHE}" \
    --limit 100 --num-workers 2


### A.2 -- volume complet (une fois le debit du pilote juge acceptable)

La resync vers Drive tourne en fond (toutes les 5 min) + un push final.
Le cache etant sauvegarde, une deconnexion se reprend en re-executant les
cellules dans l'ordre (l'etape 3 re-tire le cache, l'annotation saute le
deja-fait).

In [ ]:
start_bg_sync(L_EXTERNAL_OUT, R_EXTERNAL_OUT, every_s=300)
try:
    !.venv/bin/python apps/trainer/scripts/reannotate_external_dataset.py \
        --input-dir "{L_EXTERNAL_IN}" \
        --out-dir "{L_EXTERNAL_OUT}" \
        --cache-dir "{L_EXTERNAL_CACHE}" \
        --num-workers 2
finally:
    stop_bg_sync()
    push(L_EXTERNAL_OUT, R_EXTERNAL_OUT)


## Section B -- notre dataset (110k, deja au format interne)

### B.1 -- pilote

In [ ]:
!.venv/bin/python apps/trainer/scripts/reannotate_own_dataset.py \
    --input-dir "{L_OWN_IN}" \
    --out-dir /content/pilot_110k_bidoua \
    --cache-dir "{L_OWN_CACHE}" \
    --limit 30 --num-workers 2


### B.2 -- volume complet

In [ ]:
start_bg_sync(L_OWN_OUT, R_OWN_OUT, every_s=300)
try:
    !.venv/bin/python apps/trainer/scripts/reannotate_own_dataset.py \
        --input-dir "{L_OWN_IN}" \
        --out-dir "{L_OWN_OUT}" \
        --cache-dir "{L_OWN_CACHE}" \
        --num-workers 2
finally:
    stop_bg_sync()
    push(L_OWN_OUT, R_OWN_OUT)


## Etape suivante (pas encore decidee) -- fusion

Une fois les DEUX re-annotations terminees et poussees sur Drive
(`R_EXTERNAL_OUT` et `R_OWN_OUT`), reste a decider comment les combiner
avec le reste (`dataset_v004_290k`, `dataset_v005_400k`, pas encore
re-annotes) -- volontairement pas fait automatiquement ici, c'est une
decision a prendre une fois les deux volumes reellement re-annotes et
leurs metriques regardees (pas un merge aveugle).